# Phase-transition analysis — runs and analyses from one notebook

Observation from `heuristic_rl_rt_per_phase.ipynb` vs `llm_rt_per_phase.ipynb`:
at each phase boundary the RL agents (PPO/DDPG/MDQN) show an RT spike, the LLM agents don't.

This notebook is self-contained: it defines a set of designed load patterns, runs the benchmark for each, and analyses the results. No external scripts.

Sections:
1. Setup
2. Define scenarios (`SCENARIOS` dict)
3. Run benchmarks for selected scenarios (skip this if you only want to analyse already-collected data)
4. Analysis: load one scenario's CSVs and run all four hypothesis checks (headroom, reactivity, pre-emption, jitter)

Each scenario is designed to isolate one hypothesis — see the comments next to `SCENARIOS` below.

## 1. Setup

In [1]:
!pip install gymnasium

In [13]:
import glob
import os
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Notebook normally launches from notebooks/; the benchmark code expects
# project root as CWD (uses paths like 'configs/...', 'src/spam_cluster.py').
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
os.chdir(PROJECT_ROOT)
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'scripts', 'eval'))
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'src'))
print('CWD =', os.getcwd())

CWD = /home/vboxuser/agent-edgeautoscaling


## 2. Scenarios

Each entry is a list of phases (the same shape `benchmark_runner.run_single_benchmark` already accepts via its `load_patterns` parameter — no changes to that code were needed).

Each scenario targets ONE hypothesis from the analysis below:

| scenario           | shape                                          | tests                                                              |
|--------------------|------------------------------------------------|--------------------------------------------------------------------|
| `default`          | low(60) → medium(60) → high(60)                | original benchmark                                                 |
| `step_impulse`     | low(60) → high(60)                             | pure reactivity to a single discrete jump                          |
| `down_spike`       | high(60) → low(60) → high(60)                  | does scale-DOWN work as well as scale-up?                          |
| `oscillating`      | (low(30) → high(30)) × 3                       | adaptation across repeated transitions                             |
| `slow_ramp`        | rps 10→15→20→…→50 (20 steps each)             | removes discrete boundaries; if LLM edge disappears here, the edge WAS the jump |
| `noisy_stationary` | rps ~30 ± jitter, no real phase change         | falsifier — do agents wastefully scale on noise?                   |
| `flash_spike`      | low(50) → spike(80, 5 steps) → low(50)         | overshoot test — does a brief burst cause persistent over-provision? |


In [14]:
SCENARIOS = {
    # 'default': [
    #     {'name': 'low',    'rps': 10, 'duration_steps': 60},
    #     {'name': 'medium', 'rps': 30, 'duration_steps': 60},
    #     {'name': 'high',   'rps': 50, 'duration_steps': 60},
    # ],
    # 'step_impulse': [
    #     {'name': 'low',  'rps': 10, 'duration_steps': 60},
    #     {'name': 'high', 'rps': 50, 'duration_steps': 60},
    # ],
    # 'down_spike': [
    #     {'name': 'high', 'rps': 50, 'duration_steps': 60},
    #     {'name': 'low',  'rps': 10, 'duration_steps': 60},
    #     {'name': 'high', 'rps': 50, 'duration_steps': 60},
    # ],
    'oscillating': [
        {'name': 'low',  'rps': 10, 'duration_steps': 10},
        {'name': 'high', 'rps': 50, 'duration_steps': 10},
        {'name': 'low',  'rps': 10, 'duration_steps': 10},
        {'name': 'high', 'rps': 50, 'duration_steps': 10},
        {'name': 'low',  'rps': 10, 'duration_steps': 10},
        {'name': 'high', 'rps': 50, 'duration_steps': 10},
    ],
    'slow_ramp': [
        {'name': f'r{r}', 'rps': r, 'duration_steps': 10}
        for r in (10, 15, 20, 25, 30, 35, 40, 45, 50)
    ],
    'noisy_stationary': [
        {'name': f'n{r}', 'rps': r, 'duration_steps': 10}
        for r in (28, 34, 26, 32, 29, 31, 27, 33, 30, 31, 29, 30)
    # ],
    # 'flash_spike': [
    #     {'name': 'low',   'rps': 10, 'duration_steps': 50},
    #     {'name': 'spike', 'rps': 80, 'duration_steps':  5},
    #     {'name': 'low',   'rps': 10, 'duration_steps': 50},
    ],
}

for name, phases in SCENARIOS.items():
    total = sum(p['duration_steps'] for p in phases)
    boundaries = []
    s = 0
    for p in phases:
        s += p['duration_steps']
        boundaries.append(s)
    print(f"  {name:18s} {len(phases):>2}-phase  {total} steps  boundaries={boundaries[:-1]}")


  oscillating         6-phase  60 steps  boundaries=[10, 20, 30, 40, 50]
  slow_ramp           9-phase  90 steps  boundaries=[10, 20, 30, 40, 50, 60, 70, 80]
  noisy_stationary   12-phase  120 steps  boundaries=[10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 110]


## 3. Run benchmarks

Set `RUN_SCENARIOS` to the list of scenarios you want to execute, and `RUN_ALGORITHMS` to the algorithm names from `configs/benchmark_config.yaml`. Each scenario writes CSVs to `results/benchmark/small_<scenario>/`.

Skip this whole section if you already have data — go straight to the Analysis section.

In [15]:
# Flip to True when you actually want to launch benchmarks.
RUN = True

# Which scenarios to run (subset of SCENARIOS keys).
RUN_SCENARIOS = ['noisy_stationary', 'slow_ramp', 'oscillating']

# Which algorithms to run. None = run everything in configs/benchmark_config.yaml.
# RUN_ALGORITHMS = ['ppo', 'ddpg', 'mdqn', 'llm_vpa_ollama_mistral_latest', 'llm_vpa_ollama_llama3_8b']
RUN_ALGORITHMS = None

# Which physical scenario from configs/benchmark_config.yaml (service_count + total_resources).
PHYSICAL_SCENARIO = 'small'

# Iteration count override per scenario (None = use config['iterations']).
RUN_ITERATIONS = 3

In [ ]:
# Lazy import so the analysis-only path doesn't try to load benchmark deps.
def _import_runner():
    import importlib
    runner = importlib.import_module('benchmark_runner')
    results = importlib.import_module('benchmark_results')
    return runner, results


def run_scenario(scenario_name, load_patterns,
                 algorithms=None, physical='small', iterations=None):
    """Run one scenario end-to-end and aggregate. Mirrors the orchestration in
    `benchmark_runner.main` but takes load_patterns as a Python list instead of
    reading it from the YAML."""
    runner, results_mod = _import_runner()
    config = runner.load_benchmark_config('configs/benchmark_config.yaml')
    scen = config['scenarios'][physical]
    n_agents = scen['service_count']
    config['resources'] = scen['total_resources']

    output_dir = os.path.join(config['output_dir'], f'{physical}_{scenario_name}')
    os.makedirs(output_dir, exist_ok=True)

    algs = config['algorithms']
    if algorithms:
        algs = [a for a in algs if a['name'] in algorithms]
    algs = runner.expand_algorithms(algs)
    algs.sort(key=lambda a: 1 if a.get('apply_yaml') else 0)

    iters = iterations if iterations is not None else config['iterations']

    print(f'[{scenario_name}] {len(algs)} algorithms × {iters} iterations → {output_dir}')
    runner.reset_cluster(n_agents)
    for alg_cfg in algs:
        extra_yaml = alg_cfg.get('apply_yaml')
        if extra_yaml:
            runner.apply_yaml(extra_yaml)
        try:
            for it in range(iters):
                print(f'[{scenario_name}] {alg_cfg["name"]} iter {it}')
                runner.reset_cluster(n_agents)
                runner.run_single_benchmark(
                    alg_cfg, load_patterns, config, it, output_dir, n_agents)
        finally:
            if extra_yaml:
                runner.delete_yaml(extra_yaml)

    # Aggregate the per-iteration CSVs into summary_statistics.csv
    data = results_mod.load_all_metrics(output_dir)
    stats_df = results_mod.compute_statistics(data)
    phase_df = results_mod.per_phase_analysis(data)
    results_mod.save_results(stats_df, phase_df, output_dir)
    print(f'[{scenario_name}] done. Aggregates saved to {output_dir}/summary_statistics.csv')
    return output_dir


if RUN:
    for s in RUN_SCENARIOS:
        if s not in SCENARIOS:
            print(f'  skip {s!r} (not in SCENARIOS)')
            continue
        run_scenario(s, SCENARIOS[s], algorithms=RUN_ALGORITHMS,
                     physical=PHYSICAL_SCENARIO, iterations=RUN_ITERATIONS)
else:
    print('RUN = False — skipping benchmark execution. Flip RUN to True to launch.')

[noisy_stationary] 6 algorithms × 3 iterations → results/benchmark/small_noisy_stationary
[noisy_stationary] ppo iter 0

--- Running ppo (iteration 0) ---
Loaded model from trained/ppo/1000ep_rf_2_20rps10kepochs5alpha10epupdate50scale_a_1000resources/agent_0.pth
Loaded model from trained/ppo/1000ep_rf_2_20rps10kepochs5alpha10epupdate50scale_a_1000resources/agent_0.pth
  Phase: n28 (28 RPS, 10 steps)
Loading the cluster with 28 users on http://localhost:31416/api1/predict
Loading the cluster with 28 users on http://localhost:31416/api2/predict
  Phase: n34 (34 RPS, 10 steps)
Loading the cluster with 34 users on http://localhost:31416/api1/predict
Loading the cluster with 34 users on http://localhost:31416/api2/predict
  Phase: n26 (26 RPS, 10 steps)
Loading the cluster with 26 users on http://localhost:31416/api1/predict
Loading the cluster with 26 users on http://localhost:31416/api2/predict
  Phase: n32 (32 RPS, 10 steps)
Loading the cluster with 32 users on http://localhost:31416/api

## 4. Analysis

Point `RESULTS_DIR` at the scenario you want to inspect. The same code block below works for any scenario because transitions are auto-detected from the `rps` column.

In [ ]:
# Pick which scenario to analyse. Uncomment the one you want.
RESULTS_DIR = 'results/benchmark/small_transition'           # original (default phases)
# RESULTS_DIR = 'results/benchmark/small_step_impulse'
# RESULTS_DIR = 'results/benchmark/small_oscillating'
# RESULTS_DIR = 'results/benchmark/small_flash_spike'
# RESULTS_DIR = 'results/benchmark/small_slow_ramp'
# RESULTS_DIR = 'results/benchmark/small_noisy_stationary'
# RESULTS_DIR = 'results/benchmark/small_down_spike'

# (file_tag, display_name, group, color)
MODELS = [
    ('ppo',                       'PPO',         'RL',  '#1f77b4'),
    ('ddpg',                      'DDPG',        'RL',  '#aec7e8'),
    ('mdqn',                      'MDQN',        'RL',  '#17becf'),
    ('llm_ollama_llama3_8b',      'llama3:8b',   'LLM', '#d62728'),
    ('llm_ollama_qwen2_5_7b',     'qwen2.5:7b',  'LLM', '#ff7f0e'),
    ('llm_ollama_mistral_latest', 'mistral:7b',  'LLM', '#9467bd'),
    ('threshold',                 'Threshold',   'HEU', '#2ca02c'),
]

TRANSITIONS_OVERRIDE = None   # set to {label: step, ...} to bypass auto-detect
PRE_W  = 10
POST_W = 15

In [ ]:
def load_model(tag):
    files = sorted(glob.glob(os.path.join(RESULTS_DIR, f'{tag}_iter*_metrics.csv')))
    if not files:
        return None
    dfs = []
    for f in files:
        d = pd.read_csv(f)
        d['iter'] = int(os.path.basename(f).split('_iter')[1].split('_')[0])
        dfs.append(d)
    return pd.concat(dfs, ignore_index=True)


def parse_action(s):
    s = str(s).strip()
    if s.startswith('['):
        parts = s.strip('[]').split()
        return float(parts[0]), float(parts[1]) if len(parts) > 1 else 0.0
    try:
        return float(s), 0.0
    except Exception:
        return 0.0, 0.0


raw = {}
for tag, name, group, color in MODELS:
    d = load_model(tag)
    if d is None:
        print(f'  {name:12s} NO DATA')
        continue
    d[['vpa', 'hpa']] = pd.DataFrame(d['action'].map(parse_action).tolist(), index=d.index)
    raw[name] = {'df': d, 'group': group, 'color': color}
    print(f'  {name:12s} {group:3s}  iters={d["iter"].nunique():2d}  rows={len(d)}')

MODEL_NAMES = list(raw.keys())

In [ ]:
# Auto-detect transitions from the rps column (works for any scenario).
def detect_transitions(df):
    sub = df[df['iter'] == df['iter'].min()].sort_values('step')
    s = sub.groupby('step')['rps'].first()
    transitions = {}
    prev = s.iloc[0]
    for step, v in s.items():
        if v != prev:
            transitions[f'rps {int(prev)}→{int(v)} @ {step}'] = int(step)
            prev = v
    return transitions


if TRANSITIONS_OVERRIDE:
    TRANSITIONS = TRANSITIONS_OVERRIDE
else:
    first_df = next(iter(raw.values()))['df']
    TRANSITIONS = detect_transitions(first_df)

print(f'Detected {len(TRANSITIONS)} transition(s):')
for label, T in TRANSITIONS.items():
    print(f'  {label:30s} step={T}')

### Quantify the spike

For each (model, iter, agent, transition): `baseline_rt` = mean RT in `[T-PRE_W, T-1]`, `peak_rt` = max RT in `[T, T+POST_W]`, `spike_ratio = peak/baseline`, `recovery_steps` = steps until RT drops back below `1.2 × baseline`.

In [ ]:
def spike_stats(df, T, pre_w=PRE_W, post_w=POST_W):
    rows = []
    for (it, ag), sub in df.groupby(['iter', 'agent_id']):
        sub = sub.sort_values('step')
        pre  = sub[(sub['step'] >= T - pre_w) & (sub['step'] <  T)]['response_time']
        post = sub[(sub['step'] >= T)          & (sub['step'] <  T + post_w)]['response_time']
        if len(pre) == 0 or len(post) == 0 or pre.mean() == 0:
            continue
        baseline = pre.mean()
        peak     = post.max()
        rec = post_w
        for k, rt in enumerate(post.values):
            if rt < 1.2 * baseline:
                rec = k
                break
        rows.append({'iter': it, 'agent': ag,
                     'baseline_rt': baseline, 'peak_rt': peak,
                     'spike_ratio': peak / baseline, 'recovery_steps': rec})
    return pd.DataFrame(rows)

summary_rows = []
for name, meta in raw.items():
    for label, T in TRANSITIONS.items():
        s = spike_stats(meta['df'], T)
        if s.empty:
            continue
        summary_rows.append({
            'model': name, 'group': meta['group'], 'transition': label,
            'baseline_ms':    s['baseline_rt'].mean() * 1000,
            'peak_ms':        s['peak_rt'].mean()     * 1000,
            'spike_ratio':    s['spike_ratio'].mean(),
            'recovery_steps': s['recovery_steps'].mean(),
        })
spike_df = pd.DataFrame(summary_rows)
spike_df.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, metric, title in zip(axes,
                              ['spike_ratio', 'recovery_steps'],
                              ['Spike ratio = peak_RT / baseline_RT (1.0 = no spike)',
                               'Recovery time (steps to drop back to 1.2× baseline)']):
    pivot = spike_df.pivot(index='model', columns='transition', values=metric).reindex(MODEL_NAMES)
    pivot.plot(kind='bar', ax=ax, edgecolor='black', linewidth=0.4)
    ax.set_title(title)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(True, axis='y', alpha=0.3)
    if metric == 'spike_ratio':
        ax.axhline(1.0, color='gray', linewidth=1, linestyle='--')
plt.tight_layout()
plt.show()

### RT around each boundary

In [ ]:
W = 20
ncols = max(1, len(TRANSITIONS))
fig, axes = plt.subplots(1, ncols, figsize=(min(13, 5*ncols), 4.5), sharey=True, squeeze=False)
axes = axes[0]
for ax, (label, T) in zip(axes, TRANSITIONS.items()):
    for name, meta in raw.items():
        df = meta['df']
        sub = df[(df['step'] >= T - W) & (df['step'] < T + W)]
        g = sub.groupby('step')['response_time'].mean() * 1000
        style = '--' if meta['group'] == 'RL' else '-'
        ax.plot(g.index - T, g.values, style, color=meta['color'], label=name, linewidth=1.5, alpha=0.9)
    ax.axvline(0, color='black', linewidth=1, alpha=0.6)
    ax.set_title(f'{label}')
    ax.set_xlabel('steps relative to phase change')
    ax.set_yscale('log')
    ax.grid(True, alpha=0.3)
axes[0].set_ylabel('Mean RT (ms, log)')
axes[-1].legend(loc='center left', bbox_to_anchor=(1.02, 0.5), fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

### Headroom hypothesis
`headroom = 1 − cpu_usage / cpu_limit` in the last 3 steps before the transition. High headroom = over-provisioned = transition doesn't hurt.

In [ ]:
rows = []
for name, meta in raw.items():
    df = meta['df']
    for label, T in TRANSITIONS.items():
        pre = df[(df['step'] >= T - 3) & (df['step'] < T)].copy()
        pre['headroom'] = 1 - (pre['cpu_usage'] / pre['cpu_limit']).clip(0, 1)
        rows.append({'model': name, 'group': meta['group'], 'transition': label,
                     'cpu_limit_pre': pre['cpu_limit'].mean(),
                     'cpu_usage_pre': pre['cpu_usage'].mean(),
                     'headroom_pre':  pre['headroom'].mean()})
headroom_df = pd.DataFrame(rows)
headroom_df.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
pivot = headroom_df.pivot(index='model', columns='transition', values='headroom_pre').reindex(MODEL_NAMES)
pivot.plot(kind='bar', ax=ax, edgecolor='black', linewidth=0.4)
ax.set_ylabel('headroom = 1 − cpu_usage / cpu_limit')
ax.set_title('Pre-transition headroom (mean over last 3 steps of prior phase)')
ax.tick_params(axis='x', rotation=20)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

### Reactivity hypothesis — first scale-up *after* the boundary

In [ ]:
def first_scaleup_after(df, T, post_w=POST_W):
    out = []
    for (it, ag), sub in df.groupby(['iter', 'agent_id']):
        sub = sub.sort_values('step').set_index('step')
        if (T - 1) not in sub.index:
            continue
        base_limit = sub.loc[T - 1, 'cpu_limit']
        first = post_w
        for k in range(post_w):
            s = T + k
            if s in sub.index and sub.loc[s, 'cpu_limit'] > base_limit + 1e-3:
                first = k
                break
        out.append(first)
    return float(np.mean(out)) if out else np.nan

rows = []
for name, meta in raw.items():
    for label, T in TRANSITIONS.items():
        rows.append({'model': name, 'group': meta['group'], 'transition': label,
                     'first_scaleup_step': first_scaleup_after(meta['df'], T)})
react_df = pd.DataFrame(rows)
react_df.round(2)

### Pre-emption hypothesis — fraction of (iter, agent) pairs that already scaled up before T

In [ ]:
def preempt_frac(df, T, pre_w=PRE_W):
    flags = []
    for (it, ag), sub in df.groupby(['iter', 'agent_id']):
        sub = sub.sort_values('step').set_index('step')
        if (T - 1) not in sub.index or (T - pre_w) not in sub.index:
            continue
        flags.append(sub.loc[T - 1, 'cpu_limit'] > sub.loc[T - pre_w, 'cpu_limit'] + 1e-3)
    return float(np.mean(flags)) if flags else np.nan

rows = []
for name, meta in raw.items():
    for label, T in TRANSITIONS.items():
        rows.append({'model': name, 'group': meta['group'], 'transition': label,
                     'preempt_frac': preempt_frac(meta['df'], T)})
preempt_df = pd.DataFrame(rows)
preempt_df.round(3)

### Steady-state jitter hypothesis

In [ ]:
def steady_jitter(df):
    mask = pd.Series(True, index=df.index)
    for T in TRANSITIONS.values():
        mask &= ~df['step'].between(T - 5, T + 5)
    sub = df[mask]
    return sub.groupby(['iter', 'agent_id'])['cpu_limit'].std().mean()

jitter_rows = []
for name, meta in raw.items():
    jitter_rows.append({'model': name, 'group': meta['group'],
                        'cpu_limit_std_steady': steady_jitter(meta['df'])})
jitter_df = pd.DataFrame(jitter_rows)
jitter_df.round(2)

### Resource trajectory around each transition

In [ ]:
W = 20
n = len(raw)
ncols = max(1, len(TRANSITIONS))
fig, axes = plt.subplots(n, ncols, figsize=(min(13, 5*ncols), 2.4 * n), sharex='col', squeeze=False)

for row, (name, meta) in enumerate(raw.items()):
    df = meta['df']
    for col, (label, T) in enumerate(TRANSITIONS.items()):
        ax = axes[row, col]
        sub = df[(df['step'] >= T - W) & (df['step'] < T + W)]
        cl = sub.groupby('step')['cpu_limit'].mean()
        cu = sub.groupby('step')['cpu_usage'].mean()
        ax.plot(cl.index - T, cl.values, '--', color=meta['color'], linewidth=1.5, label='cpu_limit')
        ax.plot(cu.index - T, cu.values, '-',  color=meta['color'], linewidth=1.5, label='cpu_usage', alpha=0.9)
        ax.axvline(0, color='black', linewidth=1, alpha=0.4)
        ax.grid(True, alpha=0.3)
        if col == 0:
            ax.set_ylabel(f'{name}\n(mc)', fontsize=9)
        if row == 0:
            ax.set_title(label)
        if row == n - 1:
            ax.set_xlabel('steps rel. to T')
        if row == 0 and col == ncols - 1:
            ax.legend(loc='best', fontsize=8)
plt.tight_layout()
plt.show()

### Combined summary

In [ ]:
combo = (spike_df
         .merge(headroom_df, on=['model', 'group', 'transition'])
         .merge(react_df,    on=['model', 'group', 'transition'])
         .merge(preempt_df,  on=['model', 'group', 'transition'])
         .merge(jitter_df,   on=['model', 'group']))

cols = ['group', 'model', 'transition',
        'baseline_ms', 'peak_ms', 'spike_ratio', 'recovery_steps',
        'headroom_pre', 'first_scaleup_step', 'preempt_frac', 'cpu_limit_std_steady']
combo[cols].sort_values(['transition', 'group', 'model']).round(3)

### How to read it

- **Headroom** — LLMs over-provisioned? Look for high `headroom_pre` + low `spike_ratio` on LLM rows, vs. low headroom + high spike on RL rows.
- **Reactivity** — LLMs scaled up faster? Lower `first_scaleup_step` on LLM rows. Striking if it happens despite their large `decision_latency_ms`.
- **Pre-emption** — LLMs already scaled up before the boundary? `preempt_frac` substantially higher for LLMs than RL.
- **Steady-state jitter** — LLMs noisy in steady state and that incidentally helped? `cpu_limit_std_steady` higher for LLMs.

Run different scenarios above and watch how each metric changes:
- `step_impulse` / `flash_spike` → stresses reactivity
- `noisy_stationary` → falsifier: if LLM `cpu_limit_std_steady` blows up here even though rps barely changed, the LLM was reacting to noise, not signal
- `slow_ramp` → if RL's spike disappears, the spike was specifically about discontinuities
- `oscillating` → tests cross-iteration adaptation

## 5. Summary of findings (updated run — RL + LLMs + k8s_vpa, 20 iter × 2 agents)

Updated numbers from `phase_transition_analysis-Copy1.ipynb` (heuristic row is now `VPA = k8s_vpa`; Threshold is no longer in this analysis). The picture is **not** "LLMs handle transitions better than RL." It's three behavioural clusters separated cleanly by `headroom_pre` and `cpu_limit_std_steady`.

### Headline numbers (from the combined table)

| group | model      | low→med spike | med→high spike | low→med peak_ms | med→high peak_ms | headroom_pre (L/H) | jitter | preempt (L/H) | first_scaleup |
|-------|------------|---------------|----------------|-----------------|------------------|--------------------|--------|---------------|---------------|
| RL    | **PPO**    | **8.0×**      | **4.4×**       | **54.8**        | **29.6**         | 0.91 / 0.94        | 173    | 0.23 / 0.03   | 10.3 / 13.0   |
| RL    | DDPG       | 18×           | 22×            | 141.2           | 147.3            | 0.90 / 0.90        | 188    | 0.55 / 0.45   | **8.6 / 7.6** |
| RL    | MDQN       | 148×          | 53×            | 1091.2          | 357.7            | 0.74 / 0.89        | 168    | 0.35 / 0.33   | 10.6 / 14.0   |
| LLM   | llama3     | 6.9×          | 15.5×          | 170.9           | 193.3            | 0.88 / 0.90        | 169    | 0.35 / 0.25   | 13.1 / 11.4   |
| LLM   | **mistral**| **2.7×**      | **2.3×**       | 561.9           | 862.1            | 0.88 / 0.92        | **345**| **0.68 / 0.38**| 8.1 / 11.9   |
| LLM   | qwen       | 123×          | 5.7×           | 5006            | 16959            | 0.17 / 0.005       | 24     | 0.00 / 0.00   | 15 / 15       |
| HEU   | VPA        | 123×          | 9.6×           | 1358.7          | 5887.2           | 0.51 / 0.05        | **0**  | 0.00 / 0.00   | 15 / 15       |

### Why "spike ratio" alone is misleading

mistral and qwen each look like winners on `spike_ratio` at one transition (2.3–5.7×), but only because their `baseline_ms` is already huge (220 ms and 3826 ms respectively going into med→high). Their `peak_ms` is the *worst* in the table. The right way to read each row is the **pair** `(baseline_ms, peak_ms)`, not the ratio.

On absolute `peak_ms`, the ordering at both transitions is:

> PPO ≪ DDPG ≈ llama3 ≪ MDQN ≪ mistral ≪ VPA ≪ qwen

PPO wins outright on absolute spike. mistral's small ratio is a high-baseline artefact, not transition skill.

### Three behavioural clusters

**A. Engaged scalers** — `headroom_pre ≥ 0.7`, jitter ≥ 160, non-zero pre-emption. PPO, DDPG, MDQN, llama3, mistral. Recovery `≤ 0.73` steps at every transition.

**B. Passive failures** — `headroom_pre ≤ 0.2` on at least one side, jitter near zero, pre-emption zero, `first_scaleup_step = 15` ("never within window"). qwen and VPA.

**C. PPO** is alone at the top of cluster A on absolute peak RT.

### Hypothesis verdicts

- **Headroom** — *the dominant predictor.* Every agent with `headroom_pre ≥ 0.7` recovers cleanly; every agent with `headroom_pre < 0.3` blows up. The split is sharper than any other variable in the table and cuts across the RL/LLM/HEU labels.
- **Reactivity** — *mixed.* DDPG and mistral are fastest (~8 steps), PPO/MDQN/llama3 are in the 10–14 band, qwen and VPA never react. So *some* LLMs (mistral) actually react faster than *some* RL agents (MDQN, PPO), despite LLM decision latency. This is the most surprising update from the previous run.
- **Pre-emption** — *strongest for mistral and DDPG.* mistral pre-empts 0.68 at low→med (the highest single value in the table); DDPG 0.55. PPO barely pre-empts (0.23 / 0.03) yet still has the smallest absolute spike — its over-provisioning carries the work.
- **Steady-state jitter** — *correlated with pre-emption, not with peak RT.* mistral (345) and DDPG (188) jitter most and pre-empt most. PPO jitters moderately (173) and wins on peak anyway. qwen and VPA jitter ≈ 0 and fail.

### What `k8s_vpa` (the "VPA" row) tells us

VPA's row is the cleanest experimental control we have: it is literally the "do nothing" agent (`jitter = 0.00`, `cpu_limit_pre = 100m` at both transitions, `preempt = 0`, `first_scaleup_step = 15`). Its `headroom_pre` collapses from 0.51 at low→med to 0.047 at med→high because cpu_usage rises to fill the fixed 100 mc. The result — `peak_ms` 5887 ms — quantifies what happens with no scaling at all on this workload. **Every other agent should be compared against this floor**, not against each other in isolation.

Reminder: the reason VPA didn't actually scale here is the `vpa-updater --min-replicas=2` safety guard on single-replica deployments; see the earlier discussion. The k8s_vpa numbers in this notebook are "stock VPA defaults on single-replica deployments", not "VPA at its best".

### Bottom-line story

1. The original "LLMs beat RL at phase boundaries" observation **does not hold** as a category statement — it was driven by averaging mistral/qwen baselines that are *already* an order of magnitude above RL's, which visually compresses their transition bump.
2. **PPO wins on the metric that actually matters** (absolute peak RT at transitions): 54.8 ms and 29.6 ms. Lowest in the dataset at both boundaries.
3. **mistral is genuinely interesting** — highest jitter, highest pre-emption fraction, fastest reactivity at low→med (8.1 steps). It looks like an aggressive control policy. But its background RT is too high for that to translate into low absolute peaks. If we could combine mistral's pre-emption with PPO's baseline efficiency we'd have the best of both.
4. **qwen and stock-default VPA fail for the same structural reason**: they never raise `cpu_limit` during the run. For VPA this is the eviction-safety guard; for qwen it's apparently a function-calling/parsing failure to emit scaling actions.

### Implications for the designed scenarios (section 3)

- `noisy_stationary` is now the cleanest mistral test. Its `jitter=345` and `preempt_frac=0.68` suggest mistral might over-react to stationary noise. If `cpu_limit_std_steady` for mistral stays at ~345 under `noisy_stationary`, mistral was always being noise-driven, not phase-driven. If it drops, mistral genuinely tracks rps.
- `step_impulse` should put PPO and mistral on the same axis: PPO's over-provisioning vs. mistral's reactivity. Whichever recovers cheaper at the impulse is the real winner.
- `slow_ramp` should make PPO's lead disappear (no discontinuity), separating "PPO is over-provisioned" from "PPO handles jumps well".